In [ ]:
# Install Triton Client and DALI
# !pip install nvidia-dali-cuda120  # Matches Colab's CUDA 12.x
# !pip install tritonclient[all]
# !pip install clip

# !pip install onnxscript onnx
# !pip install --upgrade tensorrt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Triton DALI model directory structure
# !mkdir -p model_repository/preprocess/1
# mkdir -p model_repository/clip_vision/1
# mkdir -p model_repository/openclip_ensemble/1

In [135]:
import clip

In [1]:
!pip uninstall clip -y

Found existing installation: clip 1.0
Uninstalling clip-1.0:
  Successfully uninstalled clip-1.0


In [5]:
!pip install open_clip_torch

In [8]:
import clip

In [7]:
# !pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git


  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-p7mbu4ya
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-p7mbu4ya
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f5ca22695bd8ccc25a81fb0e7e1755e5af44966b066aaccd3df369ce81440c15
  Stored in directory: /tmp/pip-ephem-wheel-cache-_024ep5c/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [41]:
!ls /content/sample_data/tmp/models


In [ ]:
import torch
import clip

# 1. Download the original OpenAI model
# 'ViT-B-32' refers to the architecture
# 'openai' tells open_clip to fetch the original weights from OpenAI's servers
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)


# 2. Extract only the Vision Tower
# In OpenCLIP, the vision encoder is accessed via model.visual
vision_model = model.visual.float() # Use the Vision Tower

# 3. Create dummy input (Standard CLIP resolution is 224x224)
# Batch size 1 for the trace, but we will make it dynamic for Triton, 2 is for dynamic batch size, this was based on multiple trial and error, often the trtexec fails due hardcoded shapes duraing conversation, this ist o make sure that `2` is read `dynamic`
dummy_input = torch.randn(2, 3, 224, 224).to(device)

# 4. Export to ONNX
# Note: Using opset_version 14 for better compatibility with TensorRT 8.x+
# this is the magic porttion Opset : Operator Set - basically hardware specific instructions and optimsation by the providers. 
# For A100/H100 the opset is 17, it fuses `LayerNormalization` & `GroupNormalization` as single operators (fusted kernels): Dont forget FP8 :) for H100
# for B200  : TensorRT 10.x is recommended :: Dual-die Architectre : More on it Later!
torch.onnx.export(
    vision_model,
    dummy_input,
    "/content/sample_data/tmp/models/openai_clip_vision.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    # Essential for Triton: allows the model to handle batches 1-64
    dynamic_axes={
        'input': {0: 'batch_size'}, 
        'output': {0: 'batch_size'}
    }
)

print("Export Complete: openai_clip_vision.onnx")

/tmp/ipython-input-96707/2006346608.py:24: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0228 14:10:37.330000 96707 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0228 14:10:37.857000 96707 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' 

[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅
Applied 64 of general pattern rewrite rules.
Export Complete: openai_clip_vision.onnx


In [ ]:
# trtexec - TensorRT compiler for converting onnx to complied model, .
# !apt-get install -y libnvinfer-bin

In [ ]:
# Onnx simplifier package simplifies, what I learned is converting to onnx, certain operation can hardcode 
# shape and reshape operation can hardcode
# !pip install onnx-simplifier

In [43]:
# Simplifier, (Netron)[https://netron.app/]  helps visualizing the errors during conversion, so we have to do this particular steps, earlier version ``--dynamic-input-shape`` was a required, now its a default value
!python -m onnxsim /content/sample_data/tmp/models/openai_clip_vision.onnx /content/sample_data/tmp/models/openai_clip_vision_sim.onnx 



Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add                │ 62             │ 62               │
│ Concat             │ 32             │ 15               │
│ Constant           │ 171            │ 172              │
│ Conv               │ 1              │ 1                │
│ Expand             │ 1              │ 1                │
│ Gather             │ 37             │ 37               │
│ Gemm               │ 12             │ 12               │
│ LayerNormalization │ 26             │ 26               │
│ MatMul             │ 61             │ 61               │
│ Mul                │ 50             │ 48               │
│ Reshape            │ 135            │ 133              │
│ Shape              │ 13             │ 13               │
│ Sigmoid            │ 12             │ 12               │
│ Slice  

In [45]:
# optShapes : maximmize the performance for batch size 32,rest is self explanatory
# memPoolSize : this is to ensure to keep the memory to 2GB,  essentially used by trtexec to test different optimization tactics for each layer. it allocates this memory to run timing test to see which math operation is fastere. At the inference time it will pick 2GB over and above model wights memory and run with it.

!trtexec --onnx=/content/sample_data/tmp/models/openai_clip_vision_sim.onnx \
        --saveEngine=/content/sample_data/tmp/models/model.plan \
        --fp16 \
        --minShapes=input:1x3x224x224 \
        --optShapes=input:32x3x224x224 \
        --maxShapes=input:64x3x224x224 \
        --memPoolSize=workspace:2048MiB \
        --verbose

&&&& RUNNING TensorRT.trtexec [TensorRT v101501] [b29] # trtexec --onnx=/content/sample_data/tmp/models/openai_clip_vision_sim.onnx --saveEngine=/content/sample_data/tmp/models/model.plan --fp16 --minShapes=input:1x3x224x224 --optShapes=input:32x3x224x224 --maxShapes=input:64x3x224x224 --memPoolSize=workspace:2048MiB --verbose
[02/28/2026-14:14:09] [W] Weakly-typed networks have been deprecated in TensorRT. You can use the AutoCast tool (https://nvidia.github.io/TensorRT-Model-Optimizer/guides/8_autocast.html) to convert the network to be strongly typed.
[02/28/2026-14:14:09] [I] === Model Options ===
[02/28/2026-14:14:09] [I] Format: ONNX
[02/28/2026-14:14:09] [I] Model: /content/sample_data/tmp/models/openai_clip_vision_sim.onnx
[02/28/2026-14:14:09] [I] Output:
[02/28/2026-14:14:09] [I] === Build Options ===
[02/28/2026-14:14:09] [I] Memory Pools: workspace: 0.00195312 MiB, dlaSRAM: default, dlaLocalDRAM: default, dlaGlobalDRAM: default, tacticSharedMem: default
[02/28/2026-14:14:09

### Some brilliant insights on optimization of flags in above logs:

#### Sparsity: 
This is the famous Sparsity optimisation introduced for A100, Sparsity 2:4, however this works for sparse models, so Trt wont need it [ReadMore](https://developer.nvidia.com/blog/structured-sparsity-in-the-nvidia-ampere-architecture-and-applications-in-search-engines/)
#### Decomponsable Attentions: 
for MHA, FMHA kernel is used, for Open clip it has found a kernel that is fused, and it does not need to decompose the Matrix multiplication and other parts
#### Refit
Having this flag enabled means the weights can be swapped out without recompiling the model, we are working on inferencing so this wont be neccesary

## Other insights from the logs:

Throughput measured : 438.122 qps ***
90th percentile latency : 3.86475 ms
Host to Device : 1.57896 ms
GPU Compute Time 90th percentile(executing the kernels) : 2.27573 ms
Device to Host : 0.0128167 ms
```
02/28/2026-14:14:51] [I] === Performance summary ===
[02/28/2026-14:14:51] [I] Throughput: 438.122 qps ***
[02/28/2026-14:14:51] [I] Latency: min = 3.79248 ms, max = 4.41853 ms, mean = 3.86752 ms, median = 3.85669 ms, percentile(90%) = 3.86475 ms, percentile(95%) = 3.8678 ms, percentile(99%) = 4.41124 ms
[02/28/2026-14:14:51] [I] Enqueue Time: min = 0.450989 ms, max = 0.654297 ms, mean = 0.496265 ms, median = 0.493408 ms, percentile(90%) = 0.525879 ms, percentile(95%) = 0.537476 ms, percentile(99%) = 0.609253 ms
[02/28/2026-14:14:51] [I] H2D Latency: min = 1.57657 ms, max = 1.59998 ms, mean = 1.57896 ms, median = 1.57849 ms, percentile(90%) = 1.58057 ms, percentile(95%) = 1.58301 ms, percentile(99%) = 1.58582 ms
[02/28/2026-14:14:51] [I] GPU Compute Time: min = 2.20361 ms, max = 2.82626 ms, mean = 2.27573 ms, median = 2.26508 ms, percentile(90%) = 2.27234 ms, percentile(95%) = 2.27533 ms, percentile(99%) = 2.8201 ms
[02/28/2026-14:14:51] [I] D2H Latency: min = 0.010498 ms, max = 0.0150146 ms, mean = 0.0128167 ms, median = 0.0128174 ms, percentile(90%) = 0.013916 ms, percentile(95%) = 0.0141602 ms, percentile(99%) = 0.0145264 ms
[02/28/2026-14:14:51] [I] Total Host Walltime: 3.0083 s
[02/28/2026-14:14:51] [I] Total GPU Compute Time: 2.99942 s
[02/28/2026-14:14:51] [W] * GPU compute time is unstable, with coefficient of variance = 3.40103%.
[02/28/2026-14:14:51] [W]   If not already in use, locking GPU clock frequency or adding --useSpinWait may improve the stability.
[02/28/2026-14:14:51] [I] Explanations of the performance metrics are printed in the verbose logs.
[02/28/2026-14:14:51] [V] 
[02/28/2026-14:14:51] [V] === Explanations of the performance metrics ===
[02/28/2026-14:14:51] [V] Total Host Walltime: the host walltime from when the first query (after warmups) is enqueued to when the last query is completed.
[02/28/2026-14:14:51] [V] GPU Compute Time: the GPU latency to execute the kernels for a query.
[02/28/2026-14:14:51] [V] Total GPU Compute Time: the summation of the GPU Compute Time of all the queries. If this is significantly shorter than Total Host Walltime, the GPU may be under-utilized because of host-side overheads or data transfers.
[02/28/2026-14:14:51] [V] Throughput: the observed throughput computed by dividing the number of queries by the Total Host Walltime. If this is significantly lower than the reciprocal of GPU Compute Time, the GPU may be under-utilized because of host-side overheads or data transfers.
[02/28/2026-14:14:51] [V] Enqueue Time: the host latency to enqueue a query. If this is longer than GPU Compute Time, the GPU may be under-utilized.
[02/28/2026-14:14:51] [V] H2D Latency: the latency for host-to-device data transfers for input tensors of a single query.
[02/28/2026-14:14:51] [V] D2H Latency: the latency for device-to-host data transfers for output tensors of a single query.
[02/28/2026-14:14:51] [V] Latency: the summation of H2D Latency, GPU Compute Time, and D2H Latency. This is the latency to infer a single query.



In [48]:
!mv /content/model_repository/clip_vision/model.plan /content/model_repository/clip_vision/1/model.plan
